# Maintainer's Copilot — DistilBERT Classifier (Colab T4)

Fine-tunes `distilbert-base-uncased` for 4-class GitHub issue triage:  
`bug` / `feature` / `docs` / `question`

## Before running

1. **Runtime → Change runtime type → T4 GPU**
2. Run `python backend/scripts/download_splits.py` locally to get the split files
3. Upload these three files to Google Drive at exactly these paths:
   - `MyDrive/maintainers-copilot/splits/train.jsonl`
   - `MyDrive/maintainers-copilot/splits/val.jsonl`
   - `MyDrive/maintainers-copilot/splits/test.jsonl`
4. Run all cells top to bottom

## After training

Weights are saved to `MyDrive/maintainers-copilot/models/weights.pt`.  
Download that file locally and run `python backend/scripts/upload_weights.py` to push it to MinIO.

In [ ]:
# Cell 2 — Install dependencies
!pip install --quiet transformers torch scikit-learn

In [ ]:
# Cell 3 — Mount Google Drive and set paths
import uuid
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

DRIVE_BASE   = Path("/content/drive/MyDrive/maintainers-copilot")
SPLITS_DIR   = DRIVE_BASE / "splits"
MODELS_DIR   = DRIVE_BASE / "models"
CKPT_DIR     = MODELS_DIR / "checkpoints"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

RUN_ID = str(uuid.uuid4())
print(f"Run ID    : {RUN_ID}")
print(f"Splits dir: {SPLITS_DIR}")
print(f"Models dir: {MODELS_DIR}")

Mounted at /content/drive
Run ID    : 1f610ed8-b3a0-4a96-b301-5fe445813019
Splits dir: /content/drive/MyDrive/maintainers-copilot/splits
Models dir: /content/drive/MyDrive/maintainers-copilot/models


In [ ]:
# Cell 4 — Verify split files exist in Drive
for split in ("train", "val", "test"):
    path = SPLITS_DIR / f"{split}.jsonl"
    assert path.exists(), (
        f"Missing: {path}\n"
        "Upload the split files to MyDrive/maintainers-copilot/splits/ first."
    )
    lines = path.read_text().strip().splitlines()
    print(f"{split:<6}: {len(lines)} rows  ({path})")

train : 1307 rows  (/content/drive/MyDrive/maintainers-copilot/splits/train.jsonl)
val   : 249 rows  (/content/drive/MyDrive/maintainers-copilot/splits/val.jsonl)
test  : 311 rows  (/content/drive/MyDrive/maintainers-copilot/splits/test.jsonl)


In [ ]:
# Cell 5 — Load splits from Drive
import json

def load_jsonl(path: Path) -> list[dict]:
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]

train_data = load_jsonl(SPLITS_DIR / "train.jsonl")
val_data   = load_jsonl(SPLITS_DIR / "val.jsonl")

print(f"Train rows : {len(train_data)}")
print(f"Val rows   : {len(val_data)}")

Train rows : 1307
Val rows   : 249


In [ ]:
# Cell 6 — Class distribution and first 3 examples
from collections import Counter

LABEL2ID = {"bug": 0, "feature": 1, "docs": 2, "question": 3}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

def show_distribution(name: str, data: list[dict]) -> None:
    counts = Counter(row["label"] for row in data)
    total = len(data)
    print(f"\n{name} ({total} rows):")
    for cls in ("bug", "feature", "docs", "question"):
        n = counts.get(cls, 0)
        print(f"  {cls:<10} {n:>5}  ({100*n/total:.1f}%)")

show_distribution("train", train_data)
show_distribution("val",   val_data)

print("\n--- First 3 training examples ---")
for row in train_data[:3]:
    preview = row["text"][:120].replace("\n", " ")
    print(f"[{row['label']}] {preview} …")


train (1307 rows):
  bug          828  (63.4%)
  feature      208  (15.9%)
  docs         216  (16.5%)
  question      55  (4.2%)

val (249 rows):
  bug          138  (55.4%)
  feature       67  (26.9%)
  docs          39  (15.7%)
  question       5  (2.0%)

--- First 3 training examples ---
[question] BUG: tz_convert doesn't work for multiple timezones  ### Pandas version checks  - [X] I have checked that this issue has …
 - [X] I have checked that this  …
[bug] BUG: drop_duplicates raises unexpected error for bool[pyarrow] with null value  ### Pandas version checks  - [X] I have  …


In [ ]:
# Cell 7 — Tokenizer setup
from transformers import DistilBertTokenizerFast

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

sample = tokenizer(["Test issue title"], truncation=True, padding="max_length",
                   max_length=MAX_LENGTH, return_tensors="pt")
print(f"Tokenizer OK. Input IDs shape: {sample['input_ids'].shape}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer OK. Input IDs shape: torch.Size([1, 128])


In [ ]:
# Cell 8 — IssueDataset (torch Dataset)
import torch
from torch.utils.data import Dataset

class IssueDataset(Dataset):
    """Tokenised GitHub issues dataset for DistilBERT fine-tuning."""

    def __init__(self, rows: list[dict], label2id: dict[str, int]) -> None:
        texts  = [row["text"] for row in rows]
        labels = [label2id[row["label"]] for row in rows]
        enc = tokenizer(texts, truncation=True, padding="max_length",
                        max_length=MAX_LENGTH, return_tensors="pt")
        self.input_ids      = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]
        self.labels         = torch.tensor(labels, dtype=torch.long)

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        return {
            "input_ids":      self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels":         self.labels[idx],
        }

print("Building datasets (this may take a minute) …")
train_dataset = IssueDataset(train_data, LABEL2ID)
val_dataset   = IssueDataset(val_data,   LABEL2ID)
print(f"Train dataset: {len(train_dataset)} samples")
print(f"Val dataset  : {len(val_dataset)} samples")

Building datasets (this may take a minute) …
Train dataset: 1307 samples
Val dataset  : 249 samples


In [ ]:
# Cell 9 — Model setup
from transformers import DistilBertForSequenceClassification

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL2ID),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
model = model.to(DEVICE)
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

Device: cuda
GPU: Tesla T4


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total parameters: 66,956,548


In [ ]:
# Cell 10 — Freeze all but last transformer block + classifier head
for param in model.parameters():
    param.requires_grad = False

for param in model.distilbert.transformer.layer[5].parameters():
    param.requires_grad = True
for param in model.pre_classifier.parameters():
    param.requires_grad = True
for param in model.classifier.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f"Trainable : {trainable:,}")
print(f"Frozen    : {frozen:,}")
print(f"Fraction  : {trainable / (trainable + frozen):.1%}")

Trainable : 7,681,540
Frozen    : 59,275,008
Fraction  : 11.5%


In [ ]:
# Cell 11 — Training loop (checkpoints saved to Google Drive)
from torch.utils.data import DataLoader
from transformers import get_linear_schedule_with_warmup

BATCH_SIZE   = 32
NUM_EPOCHS   = 5
LR           = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 500
RANDOM_STATE = 42

torch.manual_seed(RANDOM_STATE)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=WEIGHT_DECAY,
)
total_steps = len(train_loader) * NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=WARMUP_STEPS, num_training_steps=total_steps,
)

print(f"Steps/epoch: {len(train_loader)}  Total: {total_steps}  Warmup: {WARMUP_STEPS}")

def save_checkpoint(epoch: int) -> None:
    path = CKPT_DIR / f"checkpoint_epoch_{epoch}_{RUN_ID}.pt"
    torch.save(model.state_dict(), path)
    print(f"  Checkpoint saved → {path}")

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for step, batch in enumerate(train_loader):
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        outputs.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += outputs.loss.item()
        if (step + 1) % 50 == 0:
            print(f"  Epoch {epoch} step {step+1}/{len(train_loader)}  "
                  f"loss={total_loss/(step+1):.4f}")

    print(f"Epoch {epoch}/{NUM_EPOCHS} — avg loss: {total_loss/len(train_loader):.4f}")
    save_checkpoint(epoch)

Steps/epoch: 41  Total: 205  Warmup: 500
Epoch 1/5 — avg loss: 1.3632
  Checkpoint saved → /content/drive/MyDrive/maintainers-copilot/models/checkpoints/checkpoint_epoch_1_1f610ed8-b3a0-4a96-b301-5fe445813019.pt
Epoch 2/5 — avg loss: 1.2935
  Checkpoint saved → /content/drive/MyDrive/maintainers-copilot/models/checkpoints/checkpoint_epoch_2_1f610ed8-b3a0-4a96-b301-5fe445813019.pt
Epoch 3/5 — avg loss: 1.1461
  Checkpoint saved → /content/drive/MyDrive/maintainers-copilot/models/checkpoints/checkpoint_epoch_3_1f610ed8-b3a0-4a96-b301-5fe445813019.pt
Epoch 4/5 — avg loss: 0.9168
  Checkpoint saved → /content/drive/MyDrive/maintainers-copilot/models/checkpoints/checkpoint_epoch_4_1f610ed8-b3a0-4a96-b301-5fe445813019.pt
Epoch 5/5 — avg loss: 0.7112
  Checkpoint saved → /content/drive/MyDrive/maintainers-copilot/models/checkpoints/checkpoint_epoch_5_1f610ed8-b3a0-4a96-b301-5fe445813019.pt


In [ ]:
# Cell 12 — Evaluate on validation set
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

def evaluate(loader: DataLoader, split_name: str) -> dict:
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            outputs = model(
                input_ids=batch["input_ids"].to(DEVICE),
                attention_mask=batch["attention_mask"].to(DEVICE),
            )
            all_preds.extend(torch.argmax(outputs.logits, dim=-1).cpu().tolist())
            all_labels.extend(batch["labels"].tolist())

    label_names = [ID2LABEL[i] for i in range(len(ID2LABEL))]
    acc      = accuracy_score(all_labels, all_preds)
    f1_macro = f1_score(all_labels, all_preds, average="macro")
    print(f"\n{'='*50}\n{split_name}\n{'='*50}")
    print(f"Accuracy : {acc:.4f}")
    print(f"Macro-F1 : {f1_macro:.4f}")
    print(classification_report(all_labels, all_preds, target_names=label_names))
    print(confusion_matrix(all_labels, all_preds))
    return {"accuracy": acc, "f1_macro": f1_macro}

val_metrics = evaluate(val_loader, "Validation")


Validation
Accuracy : 0.8755
Macro-F1 : 0.6540
              precision    recall  f1-score   support

         bug       0.86      0.97      0.91       138
     feature       0.86      0.76      0.81        67
        docs       0.94      0.85      0.89        39
    question       0.00      0.00      0.00         5

    accuracy                           0.88       249
   macro avg       0.67      0.64      0.65       249
weighted avg       0.86      0.88      0.86       249

[[134   4   0   0]
 [ 14  51   2   0]
 [  4   2  33   0]
 [  3   2   0   0]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# Cell 13 — Final evaluation on test set
test_data    = load_jsonl(SPLITS_DIR / "test.jsonl")
test_dataset = IssueDataset(test_data, LABEL2ID)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_metrics = evaluate(test_loader, "Test (held-out)")

CI_ACC = 0.70
CI_F1  = 0.65
print(f"\nCI gate: accuracy >= {CI_ACC}: {'PASS' if test_metrics['accuracy'] >= CI_ACC else 'FAIL'}")
print(f"CI gate: f1_macro >= {CI_F1}:  {'PASS' if test_metrics['f1_macro']  >= CI_F1  else 'FAIL'}")


Test (held-out)
Accuracy : 0.8939
Macro-F1 : 0.6483
              precision    recall  f1-score   support

         bug       0.89      0.97      0.93       185
     feature       0.80      0.73      0.76        51
        docs       0.98      0.84      0.91        74
    question       0.00      0.00      0.00         1

    accuracy                           0.89       311
   macro avg       0.67      0.63      0.65       311
weighted avg       0.89      0.89      0.89       311

[[179   5   1   0]
 [ 14  37   0   0]
 [  8   4  62   0]
 [  1   0   0   0]]

CI gate: accuracy >= 0.7: PASS
CI gate: f1_macro >= 0.65:  FAIL


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# Cell 14 — Save weights to Google Drive, compute SHA-256, print run summary
import hashlib

weights_path = MODELS_DIR / "weights.pt"
torch.save(model.state_dict(), weights_path)

weights_bytes  = weights_path.read_bytes()
weights_sha256 = hashlib.sha256(weights_bytes).hexdigest()

print("\n" + "="*60)
print("TRAINING COMPLETE")
print("="*60)
print(f"Run ID         : {RUN_ID}")
print(f"Weights SHA-256: {weights_sha256}")
print(f"Weights path   : {weights_path}")
print(f"Val  accuracy  : {val_metrics['accuracy']:.4f}")
print(f"Val  macro-F1  : {val_metrics['f1_macro']:.4f}")
print(f"Test accuracy  : {test_metrics['accuracy']:.4f}")
print(f"Test macro-F1  : {test_metrics['f1_macro']:.4f}")
print("="*60)
print("\nNext: download weights.pt from Google Drive, then run:")
print("  python backend/scripts/upload_weights.py")


TRAINING COMPLETE
Run ID         : 1f610ed8-b3a0-4a96-b301-5fe445813019
Weights SHA-256: 527da66c84c29cb5eeefdbd72370535a4261e9f217c27bd348c13df22013aa63
Weights path   : /content/drive/MyDrive/maintainers-copilot/models/weights.pt
Val  accuracy  : 0.8755
Val  macro-F1  : 0.6540
Test accuracy  : 0.8939
Test macro-F1  : 0.6483

Next: download weights.pt from Google Drive, then run:
  python backend/scripts/upload_weights.py


## Next steps

1. Copy **Run ID** and **SHA-256** from Cell 14 into `model_card.md`.
2. Download `weights.pt` from `MyDrive/maintainers-copilot/models/weights.pt`.
3. Place it at `backend/weights/weights.pt` locally.
4. Run `python backend/scripts/upload_weights.py` to push it to MinIO.
5. Proceed to **Phase 5** — classical ML + LLM baselines.